In [1]:
import pandas as pd
import numpy as np
from scipy.signal import argrelextrema
from scipy.signal import find_peaks
from itertools import product
from tqdm import tqdm
import time
import numba as nb
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import talib as ta
from tqdm import tqdm

In [2]:
df=pd.read_csv('../EUR_USD_2020_M5.csv')

In [3]:
df.set_index('time',inplace=True)

In [4]:
class RSI_divergence():

    def __init__(self,data):

        self.data=data

        self.params_range={'window':[10,14,20,30],
                          'up_level':[60,70,80],
                          'down_level':[20,30,40]}

    def create_params_combs(self):

        return list(product(*self.params_range.values()))

In [5]:
rsid=RSI_divergence(df)

In [6]:
len(rsid.create_params_combs())

36

In [7]:
df['RSI']=ta.RSI(df['c'],14)

In [8]:
df.iloc[50:100]

,o,h,l,c,volume,complete,RSI
time,,,,,,,
2020-01-02 02:25:00+00:00,1.12204,1.12216,1.12202,1.12216,14,True,57.081534
2020-01-02 02:30:00+00:00,1.12214,1.12222,1.12214,1.12222,8,True,59.182797
2020-01-02 02:35:00+00:00,1.12224,1.12234,1.12224,1.12231,6,True,62.174362
2020-01-02 02:40:00+00:00,1.12233,1.12239,1.12231,1.12239,9,True,64.654210
2020-01-02 02:45:00+00:00,1.12242,1.12244,1.12242,1.12244,2,True,66.147995
2020-01-02 02:50:00+00:00,1.12247,1.12247,1.12244,1.12244,4,True,66.147995
2020-01-02 02:55:00+00:00,1.12242,1.12242,1.12220,1.12224,14,True,55.305110
2020-01-02 03:00:00+00:00,1.12222,1.12227,1.12219,1.12227,11,True,56.458063
2020-01-02 03:05:00+00:00,1.12229,1.12232,1.12229,1.12230,7,True,57.634981


In [9]:
def calc_div(window, low, rsi):

    min_close=close.iloc[0]
    max_close=close.iloc[0]

    min_rsi=rsi.iloc[0]
    max_rsi=rsi.iloc[0]

    for i in range(len(close)):

        if close.iloc[i]>=max_close:
            max_close=close[i]

        if close.iloc[i]<=min_close:
            min_close=close.iloc[i]

        
    return max_close, min_close
    

In [10]:
vals=df['c'].iloc[20:30]

In [11]:
vals

time
2020-01-01 23:55:00+00:00    1.12189
2020-01-02 00:00:00+00:00    1.12192
2020-01-02 00:05:00+00:00    1.12184
2020-01-02 00:10:00+00:00    1.12162
2020-01-02 00:15:00+00:00    1.12164
2020-01-02 00:20:00+00:00    1.12162
2020-01-02 00:25:00+00:00    1.12162
2020-01-02 00:30:00+00:00    1.12166
2020-01-02 00:35:00+00:00    1.12159
2020-01-02 00:40:00+00:00    1.12175
Name: c, dtype: float64

In [12]:
slopes_vals=np.gradient(vals)

In [13]:
slopes_vals

array([ 3.0e-05, -2.5e-05, -1.5e-04, -1.0e-04,  0.0e+00, -1.0e-05,
        2.0e-05, -1.5e-05,  4.5e-05,  1.6e-04])

In [14]:
np.sum(slopes_vals)

np.float64(-4.500000000007276e-05)

In [15]:
rsi=df['RSI'].iloc[20:30]

In [16]:
slopes_rsi=np.gradient(rsi)

In [17]:
np.sum(slopes_rsi)

np.float64(-4.44937258022426)

In [18]:
rsi

time
2020-01-01 23:55:00+00:00    64.674280
2020-01-02 00:00:00+00:00    65.673589
2020-01-02 00:05:00+00:00    60.739225
2020-01-02 00:10:00+00:00    49.683860
2020-01-02 00:15:00+00:00    50.564772
2020-01-02 00:20:00+00:00    49.629052
2020-01-02 00:25:00+00:00    49.629052
2020-01-02 00:30:00+00:00    51.702173
2020-01-02 00:35:00+00:00    47.980549
2020-01-02 00:40:00+00:00    55.810352
Name: RSI, dtype: float64

In [19]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn import set_config
set_config(transform_output = "pandas")

In [20]:
rsi_vals=df.loc[:,['c','RSI']].iloc[30:80]

In [21]:
scaler_st=StandardScaler()

In [22]:
rsi_vals

,c,RSI
time,,
2020-01-02 00:45:00+00:00,1.12184,59.502823
2020-01-02 00:50:00+00:00,1.12184,59.502823
2020-01-02 00:55:00+00:00,1.12184,59.502823
2020-01-02 01:00:00+00:00,1.12218,70.954438
2020-01-02 01:05:00+00:00,1.12208,65.121687
2020-01-02 01:10:00+00:00,1.12227,70.143604
2020-01-02 01:15:00+00:00,1.12240,73.007347
2020-01-02 01:20:00+00:00,1.12238,71.865292
2020-01-02 01:25:00+00:00,1.12238,71.865292


In [23]:
st_df=scaler_st.fit_transform(rsi_vals)

In [24]:
scaler_mm=MinMaxScaler()

In [25]:
mm_df=scaler_mm.fit_transform(rsi_vals)

In [26]:
np.sum(np.gradient(rsi_vals['c'].iloc[:10]))

np.float64(0.00046500000000015973)

In [27]:
np.sum(np.gradient(rsi_vals['RSI'].iloc[:10]))

np.float64(7.6853712109998895)

In [28]:
np.sum(np.gradient(st_df['c'].iloc[:10]))

np.float64(0.9640645835711783)

In [29]:
np.sum(np.gradient(st_df['RSI'].iloc[:10]))

np.float64(0.45246094218974797)

In [30]:
np.sum(np.gradient(mm_df['c'].iloc[:10]))

np.float64(0.23724489795932868)

In [31]:
np.sum(np.gradient(mm_df['RSI'].iloc[:10]))

np.float64(0.13025401041729145)

In [32]:
from sklearn.metrics.pairwise import cosine_similarity

In [33]:
cosine_similarity([rsi_vals['c'].iloc[:10]],[rsi_vals['RSI'].iloc[:10]])

array([[0.99686742]])

In [34]:
cosine_similarity([st_df['c'].iloc[:10]],[st_df['RSI'].iloc[:10]])

array([[0.93517362]])

In [35]:
cosine_similarity([mm_df['c'].iloc[:10]],[mm_df['RSI'].iloc[:10]])

array([[0.99901505]])

In [36]:
ar1=[6,2,3,4,3,4,5,2,1,3,5,6,7,8,9,10]
ar2=[6,2,1,3,4,5,6,4,3,2,1,4,5,4,3,2]

In [37]:
ar2=[val/10 for val in ar2]

In [38]:
ar2

[0.6,
 0.2,
 0.1,
 0.3,
 0.4,
 0.5,
 0.6,
 0.4,
 0.3,
 0.2,
 0.1,
 0.4,
 0.5,
 0.4,
 0.3,
 0.2]

In [39]:
cosine_similarity([ar1],[ar2])

array([[0.83568799]])

In [40]:
np.sum(np.gradient(ar1))

np.float64(2.5)

In [41]:
np.sum(np.gradient(ar2))

np.float64(-0.6499999999999999)

In [ ]:
df_cos=df.iloc[1000:2000]

In [ ]:
cossim=[]
for i in range(len(df_cos)):

    if i>=10:

        rsi_vals=df_cos.loc[:,['c','RSI']].iloc[i-10:i]
        cossimval=cosine_similarity([rsi_vals['c']],[rsi_vals['RSI']])
        cossim.append(cossimval[0][0])

    else:

        cossim.append(np.nan)

        

In [ ]:
len(cossim)

In [ ]:
df_cos['cossim']=cossim

In [ ]:
#res = (a - a.min()) / (a.max() - a.min())
df_cos['cossim_norm']=(df_cos['cossim'] - df_cos['cossim'].min()) / (df_cos['cossim'].max() - df_cos['cossim'].min())

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
df_cos

In [ ]:
figure = make_subplots(rows=3, cols=1, row_heights=[0.5,0.25,0.25], shared_xaxes=True,vertical_spacing=0.01)
figure.update_layout(height=800, width=1200, title_text='pos_ch_retr_0.382_per_20')


figure.add_trace(go.Candlestick(x=df_cos.index,
                                open=df_cos['o'],
                                high=df_cos['h'],
                                low=df_cos['l'],
                                close=df_cos['c'],
                                name='price'), row=1, col=1)
figure.add_trace(go.Scatter(x=df_cos.index,y=df_cos['RSI'],mode='lines',line_color='blue',name='cossim'),col=1,row=2 )
figure.add_trace(go.Scatter(x=df_cos.index,y=df_cos['cossim_norm'],mode='lines',line_color='yellow',name='cossim'),col=1,row=3 )
figure.add_hline(y=0.7, col=1,row=3)

figure.update_layout(xaxis_rangeslider_visible=False)
figure.update_xaxes(
        rangebreaks=[
            dict(bounds=["sat", "mon"])]
    )
figure.show()

        